<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:




```python
import json
import pennylane as qml
import pennylane.numpy as np

class AbsMagnetization(qml.measurements.StateMeasurement):
    """A measurement class that estimates <|M|>."""

    def process_state(self, state, wire_order):
        """Calculates <|M|).

        Args:
            state (Sequence[complex]): quantum state with a flat shape. It may also have an
                optional batch dimension.

            wire_order (Wires): wires determining the subspace that the state acts on; a matrix of
                dimension 2**n that acts on a subspace of n wires

        Returns:
            abs_mag (float): <|M|)
        """
        state = np.reshape(state, -1)  # Flatten state to handle batch dimensions
        n_qubits = len(wire_order)
        Z = np.array([[1, 0], [0, -1]])

        # Calculate the magnetization for each computational basis state
        mag_values = []
        for basis_state in range(2**n_qubits):
            # Get binary representation of the basis state
            bits = np.binary_repr(basis_state, width=n_qubits)

            # Calculate the magnetization for this basis state
            mag = 0.0
            for i in range(n_qubits):
                mag += Z[int(bits[i]), int(bits[i])]
            mag /= n_qubits

            # Get the amplitude of this basis state
            amplitude = state[basis_state]
            prob = np.abs(amplitude) ** 2

            # Add to the list of values weighted by probability
            mag_values.append(prob * np.abs(mag))

        # Sum over all basis states
        abs_mag = np.sum(mag_values)
        return abs_mag

def tfim_ground_state(num_qubits, h):
    """Calculates the ground state of the 1D TFIM Hamiltonian.

    Args:
        num_qubits (int): The number of qubits / spins.
        h (float): The transverse field strength.

    Returns:
        (numpy.tensor): The ground state.
    """
    # Create the Hamiltonian
    coeffs = []
    observables = []

    # Add interaction terms Z_i Z_j for nearest neighbors in 1D chain
    for i in range(num_qubits - 1):
        coeffs.append(-1.0)
        observables.append(qml.PauliZ(i) @ qml.PauliZ(i + 1))

    # Add transverse field terms X_i
    for i in range(num_qubits):
        coeffs.append(-h)
        observables.append(qml.PauliX(i))

    H = qml.Hamiltonian(coeffs, observables)

    # Get the ground state
    eigvecs = qml.eigenvectors(H, k=1)
    ground_state = eigvecs[:, 0]

    return ground_state

dev = qml.device("default.qubit")

@qml.qnode(dev)
def magnetization(num_qubits, h):
    """Calculates the absolute value of the magnetization of the 1D TFIM
    Hamiltonian.

    Args:
        num_qubits (int): The number of qubits / spins.
        h (float): The transverse field strength.

    Returns:
        (float): <|M|.
    """
    # Calculate the ground state
    state = tfim_ground_state(num_qubits, h)

    # Load the state into the quantum device
    qml.QubitStateVector(state, wires=list(range(num_qubits)))

    return AbsMagnetization(wires=list(range(num_qubits)))

def critical_point_estimate(mags, h_values):
    """Provides a finite-size estimate of the critical point of the 1D TFIM
    Hamiltonian. The estimate is done by taking the average value of h for which
    adjacent values of <|M|> differ the most.

    Args:
        mags (numpy.tensor):
            <|M|> values for various values of h (the transverse field strength).
        h_values (numpy.tensor): The transverse field strength values.

    Returns:
        (float): The critical point estimate, h_c.
    """
    differences = [np.abs(mags[i] - mags[i + 1]) for i in range(len(mags) - 1)]
    ind = np.argmax(np.array(differences))

    h_c = np.mean([h_values[ind], h_values[ind + 1]])
    return h_c

# These functions are responsible for testing the solution.
def run(test_case_input: str) -> str:
    num_qubits = json.loads(test_case_input)
    h_values = np.arange(0.2, 1.1, 0.005)
    mags = []

    for h in h_values:
        mags.append(magnetization(num_qubits, h))

    output = critical_point_estimate(np.array(mags), h_values)

    return str(output)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)

    assert np.isclose(solution_output, expected_output, rtol=5e-3)

# These are the public test cases
test_cases = [
    ('5', '0.633'),
    ('2', '0.353')
]

# This will run the public test cases locally
for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        try:
            check(output, expected_output)
            print("Correct!")
        except AssertionError:
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")


SyntaxError: invalid syntax (3190087196.py, line 1)